# elif_ladder

`elif` ladders are how many real systems classify inputs into one of several outcomes: HTTP status groups, severity levels, pricing tiers, retry policies, and validation categories. The order of conditions is part of the logic, not just presentation.


## 1. Problem First

Why does this matter in real systems?

- API clients classify status codes into retry, fail, or ignore.
- Alerting systems map signals into severity bands.
- A wrong condition order silently sends requests down the wrong path.


In [1]:
status_code = 404

if status_code >= 500:
    category = "server_error"
elif status_code >= 400:
    category = "client_error"
elif status_code >= 300:
    category = "redirect"
else:
    category = "ok"

print(category)

client_error


## 2. Minimal Concept

Core syntax:

- `if` checks the first condition.
- `elif` checks the next condition only if earlier ones failed.
- `else` handles the fallback case.
- Exactly one branch runs in a normal `if`/`elif`/`else` chain.


## 3. Mental Model

How Python actually behaves

An `elif` ladder is a top-to-bottom first-match decision tree. Python evaluates each condition in order and stops at the first truthy one. That means correctness depends on ordering from most specific to most appropriate, not just on writing all the right conditions somewhere in the file.


In [2]:
score = 92

if score >= 90:
    grade = "A"
elif score >= 80:
    grade = "B"
elif score >= 70:
    grade = "C"
else:
    grade = "D"

print(grade)

A


## 4. Internal Mechanics

Each condition compiles into a test plus a conditional jump. Python does not evaluate the whole ladder once a match is found, which is why ordering can affect both correctness and runtime cost.


In [3]:
import dis

def classify(code):
    if code >= 500:
        return "retry"
    elif code >= 400:
        return "drop"
    else:
        return "process"

dis.dis(classify)
print(classify(503))

  3           0 RESUME                   0

  4           2 LOAD_FAST                0 (code)
              4 LOAD_CONST               1 (500)
              6 COMPARE_OP              92 (>=)
             10 POP_JUMP_IF_FALSE        1 (to 14)

  5          12 RETURN_CONST             2 ('retry')

  6     >>   14 LOAD_FAST                0 (code)
             16 LOAD_CONST               3 (400)
             18 COMPARE_OP              92 (>=)
             22 POP_JUMP_IF_FALSE        1 (to 26)

  7          24 RETURN_CONST             4 ('drop')

  9     >>   26 RETURN_CONST             5 ('process')
retry


## 5. Edge Cases

Where it breaks:

- A broad condition placed too early makes later branches unreachable.
- Overlapping ranges hide bugs until odd inputs appear.
- Missing `else` branches often leak `None` unexpectedly.


In [4]:
latency_ms = 950

if latency_ms >= 100:
    label = "slow"
elif latency_ms >= 500:
    label = "critical"
else:
    label = "healthy"

print(label)  # critical is unreachable here

slow


## 6. Performance Thinking

- Condition checks are usually O(1), but the ladder is O(k) in number of branches until a match is found.
- Put common or high-priority cases in a deliberate order.
- Expensive condition checks should not be repeated thoughtlessly.
- Sometimes a lookup table or dispatch map is cleaner than a long ladder.


## 7. Real Use Case

A pipeline classifies incoming log severity so the system can decide between storage, dashboards, and paging.


In [5]:
level = "CRITICAL"

if level == "CRITICAL":
    destination = "pager"
elif level == "ERROR":
    destination = "incident_queue"
elif level == "WARN":
    destination = "dashboard"
else:
    destination = "archive"

print(destination)

pager


## 8. Anti-Pattern

What beginners do wrong:

- Write branches in arbitrary order instead of thinking in ranges and precedence.
- Let `elif` ladders grow until they become unreadable business-rule tangles.
- Use repeated expressions instead of naming intermediate intent.


In [6]:
# Harder to audit because the most general case appears first
cpu = 87

if cpu >= 50:
    state = "busy"
elif cpu >= 80:
    state = "critical"
else:
    state = "healthy"

print(state)

busy


## 9. Interview Signals

Questions FAANG asks:

- Why does condition ordering matter in an `elif` ladder?
- When is an `elif` ladder better than separate `if` statements?
- How do you detect unreachable branches?
- When should a long ladder become a table-driven design?


## 10. Exercise (Non-trivial)

You are building response handling for an HTTP client. Requests can be processed, retried, throttled, dead-lettered, or escalated depending on exact status ranges and retry count. Design the decision flow so the order is correct, easy to test, and hard to misread in an incident.


In [7]:
def handle_response(status_code, retries):
    if status_code >= 400:
        return "fail"
    elif status_code >= 500 and retries < 3:
        return "retry"
    else:
        return "process"

# Task:
# 1. Find the ordering bug.
# 2. Refactor the ladder so specific cases win.
# 3. Explain how you would test the boundaries.